<a href="https://colab.research.google.com/github/husnanzzry/LLM-Project/blob/main/AI_Resume_Reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai gradio PyPDF2

from google import genai
from google.genai import types
import gradio as gr
from PyPDF2 import PdfReader

client = genai.Client(api_key="GEMINIAPIKEY")

def extract_text(pdf_file):
    reader = PdfReader(pdf_file.name if hasattr(pdf_file, "name") else pdf_file)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text

def review_resume(pdf_file, job_description):
    if pdf_file is None:
        return "Please upload a resume PDF first."

    if not job_description.strip():
        return "Please paste the job description first."

    resume_text = extract_text(pdf_file)

    if not resume_text.strip():
        return "Could not extract text from the PDF. Try another PDF."

    prompt = f"""
You are an experienced recruiter and ATS resume reviewer.

Based on this job description:
{job_description}

Provide this resume review:
- Strengths
- Weaknesses
- Missing Skills
- ATS Score /100
- Improvement Suggestions

Resume:
{resume_text}
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1
            )
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"

demo = gr.Interface(
    fn=review_resume,
    inputs=[
        gr.File(label="Upload Resume PDF"),
        gr.Textbox(
            label="Paste Job Description / Prompt",
            lines=10,
            placeholder="Paste the job description here..."
        )
    ],
    outputs=gr.Textbox(label="Resume Review Result", lines=20),
    title="AI Resume Reviewer",
    description="Upload a resume and paste a job description to get ATS-style feedback."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f6d459d73417e9852.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
